In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

def executar_pipeline_baseline():
    print("1. Carregando a Base de Dados Unificada...")
    df = pd.read_csv('base_unificada_asa-final.csv')

    # Remover ID do aluno se presente
    if 'ID_ALUNO' in df.columns:
        df = df.drop(columns=['ID_ALUNO'])

    # Converter colunas booleanas para numéricas
    if 'tem_bolsa' in df.columns:
        df['tem_bolsa'] = df['tem_bolsa'].astype(int)

    # 2. Pré-processamento: Codificação de variáveis categóricas (One-Hot Encoding)
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Aplicando One-Hot Encoding nas colunas: {categorical_cols}")
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    # Separar Features (X) e Alvo/Target (y)
    target_col = 'evadiu'
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # 3. Divisão em Treino e Teste (80/20 com estratificação)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Normalização de dados (recomendada para Regressão Logística e KNN)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Definição dos Modelos da Baseline
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier

    modelos = {
        "Regressão Logística": LogisticRegression(random_state=42, max_iter=1000),
        "Árvore de Decisão": DecisionTreeClassifier(random_state=42, max_depth=5),
        "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
        "KNN": KNeighborsClassifier(n_neighbors=5)
    }

    resultados = []

    print("\n--- Treinando e Avaliando Modelos ---")
    for nome, modelo in modelos.items():
        # Uso de dados escalados para LR e KNN
        if nome in ["Regressão Logística", "KNN"]:
            modelo.fit(X_train_scaled, y_train)
            y_pred = modelo.predict(X_test_scaled)
            y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
        else:
            modelo.fit(X_train, y_train)
            y_pred = modelo.predict(X_test)
            y_proba = modelo.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        try:
            roc = roc_auc_score(y_test, y_proba)
        except:
            roc = 0.0

        resultados.append({
            "Modelo": nome,
            "Acurácia": acc,
            "F1-Score (Ponderado)": f1,
            "ROC-AUC": roc
        })

        print(f"\n[{nome}]")
        print(f"Acurácia: {acc:.4f} | F1-Score: {f1:.4f} | ROC-AUC: {roc:.4f}")
        print(classification_report(y_test, y_pred))

    # Consolidar e salvar resultados
    df_resultados = pd.DataFrame(resultados)
    print("\n--- Resumo Comparativo Final ---")
    print(df_resultados.to_string(index=False))

    df_resultados.to_csv("resultados_baseline_modelos.csv", index=False)
    print("\nArquivo 'resultados_baseline_modelos.csv' gerado com sucesso!")

if __name__ == "__main__":
    executar_pipeline_baseline()

1. Carregando a Base de Dados Unificada...


FileNotFoundError: [Errno 2] No such file or directory: 'base_unificada_asa-final.csv'